# 01 — Thử nghiệm
Mỗi cell là **một run**. Kết quả tự lưu vào `Drive/DL/vnfood_colab/runs/<run_id>/`, dù chạy trên Colab hay trên máy.
- E1–E4 đã chạy trên Colab T4. Trên máy (M4) bắt đầu từ **C1** (đối chứng), rồi T1 → T3 → T2 → T4a → T4b → T5.
- Bị ngắt giữa chừng: mở lại notebook, chạy cell 0, cell 1 rồi chạy lại **đúng cell đang dở** → tự chạy tiếp từ epoch đã lưu.
- Run đã xong sẽ in `SKIP ... (DONE)`, không train lại.
- Thứ tự ưu tiên: E1 → E4 → E2 → E3 → C1 → T1 → T3 → T2 → T4a → T4b → T5 → E5.

## 0. Chuẩn bị phiên (Colab hoặc máy M4)
Chạy cell này **mỗi lần mở notebook** (kể cả sau khi bị ngắt). Cell tự nhận biết đang ở đâu:
- **Colab:** mount Drive, copy code về `/content/vnfood`, kiểm tra code khớp bản Claude vừa sync (hash) và GPU T4 (*Runtime → Change runtime type → T4 GPU*). Nếu Claude sync lại trong lúc notebook đang mở: *Runtime → Restart session* rồi chạy lại từ cell này.
- **Máy (M4):** chạy thẳng trên repo, kernel `vnfood (.venv)`, GPU `mps`. Kết quả vẫn ghi vào thư mục Drive (bật Drive for Desktop).

In [1]:
import sys
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    shutil.copytree("/content/drive/MyDrive/DL/vnfood_colab/code", "/content/vnfood", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("data", "models"))
    REPO = Path("/content/vnfood")
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "colab" / "session.py").is_file())
sys.path.insert(0, str(REPO))
from colab import session
env = session.start(REPO, ON_COLAB)
RUNS, MANIFEST, DEVICE = env["runs"], env["manifest"], env["device"]

Runs: /Users/thanhhhtt/Library/CloudStorage/GoogleDrive-hahiepthanhhhtt@gmail.com/My Drive/DL/vnfood_colab/runs
Thiết bị: mps (Apple M4 Pro)


## 1. Khôi phục data và xem trạng thái các run

In [2]:
if not session.restore_data(REPO, env["data"], manifest_dst=MANIFEST):
    raise SystemExit("Chưa có data trên Drive — chạy notebook 00 trước.")
from pathlib import Path
from colab.experiments import EXPERIMENTS, PRIORITY, control, run_state, tuning
from colab import compare
for key in PRIORITY:
    run_id = EXPERIMENTS[key].run_id if key in EXPERIMENTS else f"{key} (sau khi chốt base)"
    state = run_state(RUNS / EXPERIMENTS[key].run_id) if key in EXPERIMENTS else "-"
    print(f"{key:4s} {run_id:32s} {state}")

E1   E1_cnn                           done
E4   E4_efficientnet_b0               done
E2   E2_resnet50                      done
E3   E3_mobilenet_v3_large            done
C1   C1 (sau khi chốt base)           -
T1   T1 (sau khi chốt base)           -
T3   T3 (sau khi chốt base)           -
T2   T2 (sau khi chốt base)           -
T4a  T4a (sau khi chốt base)          -
T4b  T4b (sau khi chốt base)          -
T5   T5 (sau khi chốt base)           -
E5   E5_convnext_tiny                 new


### Chạy thử tốc độ trên máy này
Train thử 30 batch của mô hình base (không ghi gì vào `runs/`), rồi ước tính thời gian mỗi epoch và mỗi run.

In [3]:
import json
from colab import train_arch
from colab.experiments import RunConfig
pin = RUNS / "tuning_base.json"
trial = RunConfig(**json.loads(pin.read_text())["config"]) if pin.is_file() else EXPERIMENTS["E2"]
estimate = train_arch.benchmark(trial, json.loads(MANIFEST.read_text()), REPO)
print(f"{trial.run_id} trên {estimate['device']}: {estimate['seconds_per_batch']:.2f} s/batch, "
      f"~{estimate['est_epoch_minutes']:.1f} phút/epoch, ~{estimate['est_run_hours']:.1f} giờ/run (tối đa {trial.epochs} epoch)")

/Users/thanhhhtt/repos/2631/dl/final-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


E2_resnet50 trên mps (Apple M4 Pro): 0.33 s/batch, ~3.3 phút/epoch, ~0.7 giờ/run (tối đa 12 epoch)


## 2. So sánh kiến trúc (cùng split sạch, cùng công thức train)
### E1 — CNN 4 block tự xây (train từ đầu)

In [4]:
!"{sys.executable}" -m colab.train_arch --exp E1 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

SKIP E1_cnn (DONE)


### E4 — EfficientNet-B0 fine-tune (ứng viên chính)

In [5]:
!"{sys.executable}" -m colab.train_arch --exp E4 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

SKIP E4_efficientnet_b0 (DONE)


### E2 — ResNet-50 fine-tune

In [6]:
!"{sys.executable}" -m colab.train_arch --exp E2 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

SKIP E2_resnet50 (DONE)


### E3 — MobileNetV3-Large fine-tune

In [7]:
!"{sys.executable}" -m colab.train_arch --exp E3 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

SKIP E3_mobilenet_v3_large (DONE)


## 3. Chốt mô hình base cho tinh chỉnh
Chọn mô hình có **validation accuracy** cao nhất trong E1–E4 (không nhìn test). Chốt một lần, lưu `runs/tuning_base.json`.

In [8]:
base = compare.pin_tuning_base(RUNS, compare.collect(RUNS))
print("Base cho tinh chỉnh:", base.run_id)
for key, cfg in {**control(base), **tuning(base)}.items():
    print(key, cfg.run_id, run_state(RUNS / cfg.run_id))

Base cho tinh chỉnh: E2_resnet50
C1 C1_resnet50_m4 new
T1 T1_resnet50_frozen new
T2 T2_resnet50_scratch new
T3 T3_resnet50_noaug new
T4a T4a_resnet50_lr1e-3 new
T4b T4b_resnet50_lr1e-4 new
T5 T5_resnet50_ls0.1 new


## 4. Đối chứng trên máy: C1 = base chạy lại trên M4
Cấu hình **giống hệt** base, chỉ khác phần cứng (M4 `mps`, fp32 thay vì T4 fp16). Các run T* chạy trên máy được so với C1,
nên mỗi T* chỉ khác đúng một yếu tố. Hiệu số base − C1 là ảnh hưởng của phần cứng.

In [9]:
!"{sys.executable}" -m colab.train_arch --control C1 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START C1_resnet50_m4 on mps (amp=False)
Epoch 01: train_acc=0.6850 val_acc=0.8010 (213s)
Epoch 02: train_acc=0.8435 val_acc=0.8320 (196s)
Epoch 03: train_acc=0.8866 val_acc=0.8299 (201s)
Epoch 04: train_acc=0.9130 val_acc=0.8379 (202s)
Epoch 05: train_acc=0.9309 val_acc=0.8559 (203s)
Epoch 06: train_acc=0.9586 val_acc=0.8521 (205s)
Epoch 07: train_acc=0.9675 val_acc=0.8698 (206s)
Epoch 08: train_acc=0.9792 val_acc=0.8760 (207s)
Epoch 09: train_acc=0.9875 val_acc=0.8821 (207s)
Epoch 10: train_acc=0.9925 val_acc=0.8918 (206s)
Epoch 11: train_acc=0.9949 val_acc=0.8915 (208s)
Epoch 12: train_acc=0.9960 val_acc=0.8920 (213s)
DONE C1_resnet50_m4: test_acc=0.9007 macro_f1=0.9030 top3=0.9698


## 5. Tinh chỉnh (mỗi run chỉ đổi đúng một yếu tố so với mốc: C1 nếu chạy trên máy, base nếu chạy trên Colab)
### T1 — Freeze backbone, chỉ train classifier: *fine-tune toàn bộ có đáng không?*

In [10]:
!"{sys.executable}" -m colab.train_arch --tune T1 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START T1_resnet50_frozen on mps (amp=False)
Epoch 01: train_acc=0.4347 val_acc=0.5928 (76s)
Epoch 02: train_acc=0.6415 val_acc=0.6547 (67s)
Epoch 03: train_acc=0.6972 val_acc=0.6823 (70s)
Epoch 04: train_acc=0.7238 val_acc=0.7013 (82s)
Epoch 05: train_acc=0.7388 val_acc=0.7091 (86s)
Epoch 06: train_acc=0.7517 val_acc=0.7182 (83s)
Epoch 07: train_acc=0.7612 val_acc=0.7219 (80s)
Epoch 08: train_acc=0.7685 val_acc=0.7249 (79s)
Epoch 09: train_acc=0.7697 val_acc=0.7273 (79s)
Epoch 10: train_acc=0.7756 val_acc=0.7284 (78s)
Epoch 11: train_acc=0.7756 val_acc=0.7289 (78s)
Epoch 12: train_acc=0.7748 val_acc=0.7289 (78s)
DONE T1_resnet50_frozen: test_acc=0.7459 macro_f1=0.7500 top3=0.9063


### T3 — Không augmentation: *augmentation có giảm overfit không?*

In [11]:
!"{sys.executable}" -m colab.train_arch --tune T3 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START T3_resnet50_noaug on mps (amp=False)
Epoch 01: train_acc=0.6881 val_acc=0.8160 (210s)
Epoch 02: train_acc=0.8718 val_acc=0.8125 (224s)
Epoch 03: train_acc=0.9314 val_acc=0.8280 (214s)
Epoch 04: train_acc=0.9614 val_acc=0.8428 (223s)
Epoch 05: train_acc=0.9743 val_acc=0.8436 (216s)
Epoch 06: train_acc=0.9876 val_acc=0.8535 (220s)
Epoch 07: train_acc=0.9936 val_acc=0.8639 (225s)
Epoch 08: train_acc=0.9964 val_acc=0.8663 (211s)
Epoch 09: train_acc=0.9986 val_acc=0.8725 (213s)
Epoch 10: train_acc=0.9995 val_acc=0.8757 (210s)
Epoch 11: train_acc=0.9996 val_acc=0.8824 (215s)
Epoch 12: train_acc=0.9998 val_acc=0.8813 (221s)
DONE T3_resnet50_noaug: test_acc=0.8863 macro_f1=0.8873 top3=0.9634


### T2 — Không dùng ImageNet pretrained: *transfer learning giúp bao nhiêu?*

In [12]:
!"{sys.executable}" -m colab.train_arch --tune T2 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START T2_resnet50_scratch on mps (amp=False)
Epoch 01: train_acc=0.1391 val_acc=0.2138 (202s)
Epoch 02: train_acc=0.2634 val_acc=0.3081 (219s)
Epoch 03: train_acc=0.3473 val_acc=0.3458 (222s)
Epoch 04: train_acc=0.4157 val_acc=0.3927 (218s)
Epoch 05: train_acc=0.4725 val_acc=0.4265 (211s)
Epoch 06: train_acc=0.5246 val_acc=0.5272 (219s)
Epoch 07: train_acc=0.5720 val_acc=0.5205 (209s)
Epoch 08: train_acc=0.6291 val_acc=0.5342 (214s)
Epoch 09: train_acc=0.6812 val_acc=0.6153 (210s)
Epoch 10: train_acc=0.7288 val_acc=0.6330 (207s)
Epoch 11: train_acc=0.7638 val_acc=0.6649 (208s)
Epoch 12: train_acc=0.7862 val_acc=0.6700 (224s)
DONE T2_resnet50_scratch: test_acc=0.6818 macro_f1=0.6752 top3=0.8625


### T4a / T4b — Learning rate 1e-3 và 1e-4 (base 3e-4)

In [13]:
!"{sys.executable}" -m colab.train_arch --tune T4a --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START T4a_resnet50_lr1e-3 on mps (amp=False)
Epoch 01: train_acc=0.6168 val_acc=0.6796 (202s)
Epoch 02: train_acc=0.7638 val_acc=0.7206 (198s)
Epoch 03: train_acc=0.8098 val_acc=0.7420 (202s)
Epoch 04: train_acc=0.8448 val_acc=0.7758 (201s)
Epoch 05: train_acc=0.8736 val_acc=0.8079 (214s)
Epoch 06: train_acc=0.9043 val_acc=0.8079 (222s)
Epoch 07: train_acc=0.9273 val_acc=0.8473 (222s)
Epoch 08: train_acc=0.9516 val_acc=0.8540 (224s)
Epoch 09: train_acc=0.9708 val_acc=0.8634 (204s)
Epoch 10: train_acc=0.9813 val_acc=0.8722 (222s)
Epoch 11: train_acc=0.9884 val_acc=0.8765 (214s)
Epoch 12: train_acc=0.9922 val_acc=0.8776 (215s)
DONE T4a_resnet50_lr1e-3: test_acc=0.8783 macro_f1=0.8778 top3=0.9656


In [14]:
!"{sys.executable}" -m colab.train_arch --tune T4b --runs-dir "{RUNS}" --manifest "{MANIFEST}"

START T4b_resnet50_lr1e-4 on mps (amp=False)
Epoch 01: train_acc=0.6385 val_acc=0.8286 (210s)
Epoch 02: train_acc=0.8607 val_acc=0.8476 (220s)
Epoch 03: train_acc=0.9116 val_acc=0.8615 (221s)
Epoch 04: train_acc=0.9391 val_acc=0.8669 (217s)
Epoch 05: train_acc=0.9558 val_acc=0.8741 (219s)
Epoch 06: train_acc=0.9719 val_acc=0.8720 (217s)
Epoch 07: train_acc=0.9788 val_acc=0.8797 (217s)
Epoch 08: train_acc=0.9867 val_acc=0.8843 (218s)
Epoch 09: train_acc=0.9913 val_acc=0.8875 (221s)
Epoch 10: train_acc=0.9927 val_acc=0.8899 (219s)
Epoch 11: train_acc=0.9947 val_acc=0.8891 (219s)
Epoch 12: train_acc=0.9948 val_acc=0.8907 (218s)
DONE T4b_resnet50_lr1e-4: test_acc=0.8914 macro_f1=0.8930 top3=0.9680


### T5 — Label smoothing 0.1

In [3]:
!"{sys.executable}" -m colab.train_arch --tune T5 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

RESUME T5_resnet50_ls0.1 from epoch 5; best val=0.8411
Epoch 05: train_acc=0.9586 val_acc=0.8666 (231s)
Epoch 06: train_acc=0.9845 val_acc=0.8706 (224s)
Epoch 07: train_acc=0.9922 val_acc=0.8746 (216s)
Epoch 08: train_acc=0.9956 val_acc=0.8803 (216s)
Epoch 09: train_acc=0.9833 val_acc=0.8765 (229s)
Epoch 10: train_acc=0.9910 val_acc=0.8870 (229s)
Epoch 11: train_acc=0.9941 val_acc=0.8928 (225s)
Epoch 12: train_acc=0.9967 val_acc=0.8912 (222s)
DONE T5_resnet50_ls0.1: test_acc=0.8865 macro_f1=0.8879 top3=0.9632


## 6. Tùy chọn (nếu còn quota)
### E5 — ConvNeXt-Tiny fine-tune

In [ ]:
!"{sys.executable}" -m colab.train_arch --exp E5 --runs-dir "{RUNS}" --manifest "{MANIFEST}"

### Multi-seed cho base (đo độ ổn định: seed 123 và 2024)

In [ ]:
!"{sys.executable}" -m colab.train_arch --seed-run S123 --runs-dir "{RUNS}" --manifest "{MANIFEST}"
!"{sys.executable}" -m colab.train_arch --seed-run S2024 --runs-dir "{RUNS}" --manifest "{MANIFEST}"